# Bulk RNA-Seq Analysis

### Prerequisites

#### Software and Dependencies

- **sra-tools:** NCBI's SRA Toolkit for downloading and processing data from the Sequence Read Archive. Specifically, prefetch and fasterq-dump are used here.
- **eutils:** E-Utilities are a set of NCBI web services that allow you to search, retrieve, and manage data from Entrez databases using simple URLs with query parameters
- **Trimmomatic:** Used for quality trimming of raw sequencing reads.
- **FastQC:** A quality control tool for high-throughput sequence data.
- **MultiQC:** Aggregates results from multiple FastQC runs into a single report.
- **Salmon:** A tool for quantifying transcript abundances from RNA-Seq data.
- **gffread:** Parses GFF/GTF annotation files and extracts information from them, such as to create a transcriptome reference file from genome and annotation files.


### STEP 1: Install the tools

Checking conda environment

In [ ]:
!conda info

In [ ]:
%conda install mamba -c conda-forge -y

In [ ]:
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict

In [ ]:
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

In [ ]:
!conda install -c bioconda trimmomatic fastqc multiqc salmon eutils gffread parallel-fastq-dump sra-tools=3.0.5 pigz -q -y 

In [ ]:
import sys
import os

# Find the bin folder of your current environment
env_bin = os.path.join(sys.prefix, 'bin')
print(f"Your tools are located in: {env_bin}")

In [ ]:
!/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/sratoolkit.3.4.1-ubuntu64/bin/prefetch --version

### STEP 2: Setup Environment


Create a set of directories to store the reads, reference sequence files, and output files. 

In [ ]:
!pwd
CORES = 4

In [ ]:
! mkdir -p data
! mkdir -p data/raw_fastq
! mkdir -p data/trimmed
! mkdir -p data/fastqc
! mkdir -p data/reference
! mkdir -p data/quants

### STEP 3: Downloading relevant SRA files using SRA Tools

Next we will need to download the relevant fastq files.

Because these files can be large, the process of downloading and extracting fastq files can be quite lengthy.

The sequence data for this tutorial comes from work by Cushman et al., [Increased whiB7 expression and antibiotic resistance in Mycobacterium chelonae carrying two prophages.](https://link.springer.com/article/10.1186/s12866-021-02224-z)

It provides evidence of increased antibiotic resistance and expression of intrinsic antibiotic resistance genes in a strain of Mycobacterium chelonae carrying prophage. Strains carrying the prophage McProf demonstrated increased resistance to amikacin. Resistance in these strains was further enhanced by exposure to sub-inhibitory concentrations of the antibiotic, acivicin, or by the presence of a second prophage, BPs. Increased expression of the virulence gene, whiB7, was observed in strains carrying both prophages, BPs and McProf, relative to strains carrying a single prophage or no prophages.

We will be downloading the sample runs from this project using SRA tools, downloading from the NCBI's SRA (Sequence Run Archives).

However, first we need to find the associated accession numbers in order to download.

### STEP 3.1: Finding run accession numbers.

The SRA stores sequence data in terms of runs, (SRR stands for Sequence Read Run). To download runs, we will need the accession ID for each run we wish to download.

The Cushman et al., project contains 12 runs. To make it easier, these are the run IDs associated with this project:
SRR13349122
SRR13349123
SRR13349124
SRR13349125
SRR13349126
SRR13349127
SRR13349128
SRR13349129
SRR13349130
SRR13349131
SRR13349132
SRR13349133
In this case, all these runs belong to the SRP (Sequence Run Project): SRP300216.

Sequence run experiments can be searched for using the SRA database on the NCBI website; and article-specific sample run information can be found in the supplementary section of that article.

For instance, here, the the authors posted a link to the sequence data GSE (Gene Series number), [GSE164210](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE164210). This leads to the appropriate 'Gene Expression Omnibus' page where, among other useful files and information, the relevant SRA database link can be found.

### STEP 3.2: Using the SRA-toolkit for a single sample.

In [ ]:
! /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/sratoolkit.3.4.1-ubuntu64/bin/prefetch SRR13349122 -O /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq -f yes

Notice the SRA archives sequence files in the SRA format.

Typically genome workflows process data in the form of zipped or unzipped .fastq, or .fasta files

So before we move on, we need to convert the files from .sra to .fastq using the fastq-dump tool.

We will also compresss the fastq files to make them take less space, making them fastq.gz files.

### STEP 3.3: Downloading multiple files using the SRA-toolkit.

In [ ]:
sra_list = [
    "SRR13349122",
    "SRR13349123",
    "SRR13349124"
]

for sra in sra_list:
    ! /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/sratoolkit.3.4.1-ubuntu64/bin/prefetch {sra} -O data/raw_fastq -f yes


### STEP 3.4: Converting Multiple SRA files to Fastq and convert to fastq.gz

In [ ]:
%%bash
# 1. Your exact main directory where the subfolders live
PARENT_DIR="/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq"

# 2. Find and convert each nested .sra file dynamically
find "$PARENT_DIR" -type f -name "*.sra" | while read -r sra_file; do
    # This automatically finds the unique subfolder for this specific file
    TARGET_DIR=$(dirname "$sra_file")
    FILE_NAME=$(basename "$sra_file")
    
    echo "=================================================="
    echo "Processing: $FILE_NAME inside $TARGET_DIR"
    echo "=================================================="
    
    # 3. Your exact code, but with $TARGET_DIR instead of a hardcoded path
    /anaconda/envs/azureml_py310_sdkv2/bin/fasterq-dump "$sra_file" \
        --split-files \
        --threads 8 \
        -O "$TARGET_DIR" \
        -f
        
    # 4. Immediately compress to save server disk space
    echo "Compressing FASTQ files for $FILE_NAME..."
    gzip "${TARGET_DIR}"/*.fastq
done

echo "=================================================="
echo "All files processed and compressed successfully!"
echo "=================================================="


Processing: SRR13349122.sra inside /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349122

spots read      : 10,827,590
reads read      : 21,655,180
reads written   : 21,655,180
Compressing FASTQ files for SRR13349122.sra...

Processing: SRR13349123.sra inside /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349123

spots read      : 11,165,256
reads read      : 22,330,512
reads written   : 22,330,512
Compressing FASTQ files for SRR13349123.sra...

### STEP 4: Copy reference transcriptome files that will be used by Salmon using eu

Salmon is a tool that aligns RNA-Seq reads to a transcriptome.

So we will need a transcriptome reference file.

To get one, we can search through the NCBI assembly database, find an assembly, and download transcriptome reference files from that assembly using FTP links.

For instance, we will use the [ASM163280v1](https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_001632805.1/) refseq assembly, found by searching through the NCBI assembly database. The FTP links can be accessed through the website in various ways, one way is to click the 'FTP directory for RefSeq assembly' link, found under 'Access the data', section.

Alternatively, if one were inclined, one could take the less common route and perform this through the NCBI command line tool suite called 'Entrez Direct' (EDirect).

This is an intricate and complicated set of tools, with many ways to do any one thing.

Below is an example of using an eDirect search query with a refseq identifier to obtain the relevant FTP directory, and then using that to download desired reference files.

In [ ]:

# ============================================================
#  Download ALL common NCBI assembly files for an accession
#  (Genome FASTA, GFF3, Protein FASTA, CDS FASTA, RNA FASTA)
#  Saves into your existing "reference/" folder.
# ============================================================

import os
import re
import time
import hashlib
import requests
from typing import Dict, Optional, Tuple
from urllib.parse import urljoin

EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"  # Official base URL

# Exact filename suffixes for each format (per NCBI conventions)
SUFFIXES = {
    "genome":  "_genomic.fna.gz",           # genome FASTA
    "gff":     "_genomic.gff.gz",           # GFF3 annotation
    "protein": "_protein.faa.gz",           # protein FASTA
    "cds":     "_cds_from_genomic.fna.gz",  # CDS nucleotide FASTA
    "rna":     "_rna_from_genomic.fna.gz",  # RNA FASTA (derived)
}


def resolve_uid(accession: str, timeout: int = 30) -> str:
    """Resolve an Assembly UID for a given accession."""
    r = requests.post(
        EUTILS_BASE + "esearch.fcgi",
        data={"db": "assembly", "term": accession, "retmax": 1, "usehistory": "y"},
        timeout=timeout,
    )
    r.raise_for_status()
    m = re.search(r"<Id>(\d+)</Id>", r.text)
    if not m:
        raise RuntimeError(f"Could not resolve UID for {accession}")
    return m.group(1)


def get_https_dir(uid: str, prefer: str = "refseq", timeout: int = 30) -> Tuple[str, str]:
    """Return (https_dir, folder_base) from esummary; prefer 'refseq' or 'genbank'."""
    r = requests.post(
        EUTILS_BASE + "esummary.fcgi",
        data={"db": "assembly", "id": uid, "version": "2.0", "retmode": "xml"},
        timeout=timeout,
    )
    r.raise_for_status()
    refseq = re.search(r"<FtpPath_RefSeq>([^<]+)</FtpPath_RefSeq>", r.text)
    genbank = re.search(r"<FtpPath_GenBank>([^<]+)</FtpPath_GenBank>", r.text)

    ftp = None
    if prefer.lower() == "refseq" and refseq:
        ftp = refseq.group(1).strip()
    elif prefer.lower() == "genbank" and genbank:
        ftp = genbank.group(1).strip()
    else:
        ftp = (refseq or genbank)
        if ftp:
            ftp = ftp.group(1).strip()
    if not ftp:
        raise RuntimeError("No FTP path found in assembly summary (neither RefSeq nor GenBank).")

    https_dir = ftp.replace("ftp://", "https://")
    folder_base = https_dir.rstrip("/").split("/")[-1]
    return https_dir, folder_base

    
def fetch_md5checksums(https_dir: str, timeout: int = 60) -> Dict[str, str]:
    """
    Download md5checksums.txt and parse it into {filename -> md5}.
    Returns empty dict if the file isn't available.
    """
    url = urljoin(https_dir + "/", "md5checksums.txt")
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code != 200:
            return {}
        lines = r.text.splitlines()
        md5map = {}
        for line in lines:
            # format: MD5 (optional) followed by spaces and filename; guard for various layouts
            m = re.search(r"([a-f0-9]{32})\s+(.+)$", line.strip())
            if m:
                md5map[m.group(2).strip()] = m.group(1)
        return md5map
    except Exception:
        return {}


def select_remote_filename(md5map: Dict[str, str], suffix: str, folder_base: str) -> str:
    """
    Find the correct remote filename for a given suffix by scanning md5checksums.txt.
    Falls back to constructing folder_base + suffix if not found.
    """
    # Scan for an exact suffix match first
    for fname in md5map.keys():
        if fname.endswith(suffix):
            # Guard: avoid accidental matches (e.g., ensure exact suffix end)
            return fname
    # Fallback: construct conventional name
    return f"{folder_base}{suffix}"


def download_stream(url: str, dest: str, retries: int = 5, timeout: int = 90) -> bool:
    """Stream download with simple exponential backoff and atomic replace."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=timeout, headers={
                "User-Agent": "NCBI-Downloader/1.0 (+contact@example.com)"
            }) as r:
                if r.status_code == 404:
                    return False
                r.raise_for_status()
                tmp = dest + ".part"
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
                os.replace(tmp, dest)
                return True
        except Exception as e:
            print(f" {os.path.basename(dest)}: attempt {attempt}/{retries} failed: {e}")

            time.sleep(2 * attempt)
    return False


def md5sum(path: str) -> str:
    """Compute MD5 for a local file."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_assembly_all(
    accession: str,
    out_dir: str = "reference",
    include: Tuple[str, ...] = ("genome", "gff", "protein", "cds", "rna"),
    prefer: str = "refseq",
    verify_md5: bool = False,
) -> Dict[str, Optional[str]]:
  
    # Resolve location
    uid = resolve_uid(accession)
    https_dir, folder_base = get_https_dir(uid, prefer=prefer)

    # Get md5 index (best effort)
    md5map = fetch_md5checksums(https_dir)
    results: Dict[str, Optional[str]] = {}
    for filetype in include:
        if filetype not in SUFFIXES:
            print(f" Unknown filetype '{filetype}'. Skipping.")
            results[filetype] = None
            continue

        suffix = SUFFIXES[filetype]
        remote_name = select_remote_filename(md5map, suffix, folder_base)
        url = urljoin(https_dir + "/", remote_name)
        local_path = os.path.join(out_dir, os.path.basename(remote_name))

        print(f"  {filetype}: {url}")
        ok = download_stream(url, local_path)
        if not ok:
            print(f"{filetype}: not found on server (skipped).")
            results[filetype] = None
            continue

        if verify_md5 and md5map:
            expected = md5map.get(remote_name)
            if expected:

                got = md5sum(local_path)
                if got.lower() != expected.lower():
                    print(f"MD5 mismatch for {filetype}: expected {expected}, got {got}")
                else:
                    print(f"MD5 verified for {filetype}")

        results[filetype] = local_path

    print("\n Done.\nSummary:", results)
    return results


In [ ]:
# This will save files into your existing "reference" folder.
# You can adjust 'include' to choose a subset.
results = download_assembly_all("GCF_001632805.1", out_dir="/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference",
                                include=("genome", "gff", "protein", "cds", "rna"),
                                prefer="refseq", verify_md5=True)

In [ ]:
#unzip the compresseed fasta files
! gzip -d data/reference/*.gz --force

Next we can use a tool called gffread to create a transcriptome reference file using the gtf and genome files we downloaded.

In [ ]:
! /anaconda/envs/azureml_py310_sdkv2/bin/gffread -w /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_transcriptome_reference.fa -g /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_ASM163280v1_genomic.fna /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_ASM163280v1_genomic.gff

It is also recommended to include the full genome at the end of the transcriptome reference file, for the purpose of performing a 'decoy-aware' mapping, more information about which can be found in the Salmon documentation.

To alert the tool to the presence of this, we will also create a 'decoy file', which salmon needs pointed towards the full genome sequence in our transcriptome reference file.

In [ ]:
! cat /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_transcriptome_reference.fa <(echo) /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_ASM163280v1_genomic.fna > /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_transcriptome_reference_w_decoy.fa
! echo "NZ_CP007220.1" > /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/decoys.txt

### STEP 5:  Copy data file for Trimmomatic

One of trimmomatics functions is to trim sequence machine specific adapter sequences. These are usually within the trimmomatic installation directory in a folder called adapters.

Directories of packages within mamba installations can be confusing, so in the case of using mamba with trimmomatic, it may be easier to simply download or create a file with the relevant adapter sequencecs and store it in an easy to find directory.

In [ ]:
# Download from public GCS URL to current notebook directory
! curl -L -o TruSeq3-PE.fa \
  https://storage.googleapis.com/nigms-sandbox/me-inbre-rnaseq-pipelinev2/config/TruSeq3-PE.fa
# Inspect the top lines
! head TruSeq3-PE.fa

### STEP 6: Run Trimmomatic

Trimmomatic will trim off any adapter sequences or low quality sequence it detects in the FASTQ files.

Using piping and our original list, it is possible to queue up a batch run of trimmomatic for all our files, note that this is a different way to run a loop compared with what we did before.

The below code may take approximately 35 minutes to run.

In [ ]:
import os
import subprocess

RAW_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq"
TRIM_DIR = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed"

# Ensure output directory exists
os.makedirs(TRIM_DIR, exist_ok=True)

# List all SRR subdirectories manually (no glob)
srr_dirs = [
    d for d in os.listdir(RAW_DIR)
    if os.path.isdir(os.path.join(RAW_DIR, d)) and d.startswith("SRR")
]

print("Found SRR directories:", srr_dirs)

for ID in srr_dirs:
    sample_dir = os.path.join(RAW_DIR, ID)

    r1 = os.path.join(sample_dir, f"{ID}.sra_1.fastq.gz")
    r2 = os.path.join(sample_dir, f"{ID}.sra_2.fastq.gz")

    # Output files
    out_r1_paired = os.path.join(TRIM_DIR, f"{ID}_1_trimmed.fastq.gz")
    out_r1_unpaired = os.path.join(TRIM_DIR, f"{ID}_1_trimmed_unpaired.fastq.gz")
    out_r2_paired = os.path.join(TRIM_DIR, f"{ID}_2_trimmed.fastq.gz")
    out_r2_unpaired = os.path.join(TRIM_DIR, f"{ID}_2_trimmed_unpaired.fastq.gz")

    cmd = [
        "/anaconda/envs/azureml_py310_sdkv2/bin/trimmomatic", "PE",
        "-threads", str(CORES),
        r1, r2,
        out_r1_paired,
        out_r1_unpaired,
        out_r2_paired,
        out_r2_unpaired,
        "ILLUMINACLIP:TruSeq3-PE.fa:2:30:10:2:keepBothReads",
        "LEADING:3",
        "TRAILING:3",
        "MINLEN:36"
    ]

    print("\nRunning:", " ".join(cmd))
    subprocess.run(cmd, check=True)

print("\nAll SRR samples processed successfully.")


Found SRR directories: ['SRR13349122', 'SRR13349123', 'SRR13349124', 'SRR13349125', 'SRR13349126', 'SRR13349127', 'SRR13349128', 'SRR13349129', 'SRR13349130', 'SRR13349131', 'SRR13349132', 'SRR13349133']

Running: /anaconda/envs/azureml_py310_sdkv2/bin/trimmomatic PE -threads 4 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349122/SRR13349122.sra_1.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349122/SRR13349122.sra_2.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed_unpaired.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed_unpaired.fastq.gz ILLUMINACLIP:TruSeq3-PE.fa:2:30:10:2:keepBothReads LEADING:3 TRAILING:3 MINLEN:36
TrimmomaticPE: Started with arguments:
 -threads 4 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349122/SRR13349122.sra_1.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349122/SRR13349122.sra_2.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed_unpaired.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed_unpaired.fastq.gz ILLUMINACLIP:TruSeq3-PE.fa:2:30:10:2:keepBothReads LEADING:3 TRAILING:3 MINLEN:36
ILLUMINACLIP: Using adapter file from Trimmomatic installation folder: /anaconda/envs/azureml_py310_sdkv2/share/trimmomatic-0.40-0/adapters/TruSeq3-PE.fa
Using PrefixPair: 'TACACTCTTTCCCTACACGACGCTCTTCCGATCT' and 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences
Quality encoding detected as phred33
Input Read Pairs: 10827590 Both Surviving: 10810267 (99.84%) Forward Only Surviving: 17297 (0.16%) Reverse Only Surviving: 0 (0.00%) Dropped: 26 (0.00%)
TrimmomaticPE: Completed successfully
TrimmomaticPE: Started with arguments:
 -threads 4 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349123/SRR13349123.sra_1.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/raw_fastq/SRR13349123/SRR13349123.sra_2.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_1_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_1_trimmed_unpaired.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_2_trimmed.fastq.gz /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_2_trimmed_unpaired.fastq.gz ILLUMINACLIP:TruSeq3-PE.fa:2:30:10:2:keepBothReads LEADING:3 TRAILING:3 MINLEN:36
ILLUMINACLIP: Using adapter file from Trimmomatic installation folder: /anaconda/envs/azureml_py310_sdkv2/share/trimmomatic-0.40-0/adapters/TruSeq3-PE.fa
Using PrefixPair: 'TACACTCTTTCCCTACACGACGCTCTTCCGATCT' and 'GTGACTGGAGTTCAGACGTGTGCTCTTCCGATCT'
ILLUMINACLIP: Using 1 prefix pairs, 0 forward/reverse sequences, 0 forward only sequences, 0 reverse only sequences

### STEP 7: Run FastQC

In [ ]:
import subprocess

trim_dir = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed"
out_dir = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/fastqc"

# Loop through directory entries

for fname in os.listdir(trim_dir):
    if fname.endswith("_1_trimmed.fastq.gz"):
        base = fname.replace("_1_trimmed.fastq.gz", "")
        
        r1 = os.path.join(trim_dir, f"{base}_1_trimmed.fastq.gz")
        r2 = os.path.join(trim_dir, f"{base}_2_trimmed.fastq.gz")

        print("Running FastQC on:", r1, "and", r2)
        cmd = [
            "/anaconda/envs/azureml_py310_sdkv2/bin/fastqc",
            "-t", str(CORES),
            r1,
            r2,
            "-o", out_dir]


Running FastQC on: /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed.fastq.gz and /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed.fastq.gz
Running FastQC on: /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_1_trimmed.fastq.gz and /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349123_2_trimmed.fastq.gz
Running FastQC on: /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349124_1_trimmed.fastq.gz and /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349124_2_trimmed.fastq.gz

### STEP 8: Run MultiQC

In [ ]:
! /anaconda/envs/azureml_py310_sdkv2/bin/multiqc -f /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/fastqc/

### STEP 9: Index the Transcriptome so that Trimmed Reads Can Be Mapped Using Salmon

In [ ]:
! /anaconda/envs/azureml_py310_sdkv2/bin/salmon index -t /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/GCF_001632805.1_transcriptome_reference_w_decoy.fa -p $CORES -i /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index --decoys /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/decoys.txt -k 31 --keepDuplicates

2026-08-04T14:22:35.393484Z  INFO salmon_index: detected 1 exact-sequence-duplicate transcripts; retained them (--keepDuplicates) and recorded them in duplicate_clusters.tsv
2026-08-04T14:22:35.436099Z  INFO salmon_index: building compacted dBG (k=31, threads=4)
2026-08-04T14:22:35.468477Z  INFO cf1_rs::pipeline: RSS at start: current=52 MB, peak=110 MB
2026-08-04T14:22:35.468506Z  INFO cf1_rs::pipeline: Phase 1: Counting minimizer frequencies...
2026-08-04T14:22:35.468515Z  INFO cf1_rs::minimizer: Counting minimizer frequencies with m=15, w=17, k=31
2026-08-04T14:22:35.719886Z  INFO cf1_rs::minimizer: Histogram: 1065983 total minimizer occurrences across 65517 non-empty buckets
2026-08-04T14:22:35.726853Z  INFO cf1_rs::pipeline: RSS after P1: current=76 MB, peak=110 MB
2026-08-04T14:22:35.726869Z  INFO cf1_rs::pipeline: Phase 2: Partitioning minimizers and routing super k-mers...
2026-08-04T14:22:35.726972Z  INFO cf1_rs::minimizer: Partitioned into 128 bins (target 8327/bin)
2026-08-04T14:22:35.727058Z  INFO cf1_rs::superkmer: Routing super k-mers to 128 bins with m=15, w=17

### STEP 10: Run Salmon to Map Reads to Transcripts and Quantify Expression Levels

Salmon aligns the trimmed reads to the reference transcriptome and generates the read counts per transcript. In this analysis, each gene has a single transcript.

In [ ]:

trim_dir = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed"
index = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index"
out_base = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants"

os.makedirs(out_base, exist_ok=True)

# Loop through directory contents, find all *_1_trimmed.fastq.gz files
for filename in os.listdir(trim_dir):
    if filename.endswith("_1_trimmed.fastq.gz"):
        sample_id = filename.replace("_1_trimmed.fastq.gz", "")

        r1 = os.path.join(trim_dir, f"{sample_id}_1_trimmed.fastq.gz")
        r2 = os.path.join(trim_dir, f"{sample_id}_2_trimmed.fastq.gz")
        outdir = os.path.join(out_base, f"{sample_id}_quant")
        
        cmd = [
            "/anaconda/envs/azureml_py310_sdkv2/bin/salmon", "quant",
            "-i", index,
            "-l", "ISR",
            "-1", r1,
            "-2", r2,
            "-p", str(CORES),
            "--validateMappings",
            "-o", outdir
        ]

        print("Running:", " ".join(cmd))
        subprocess.run(cmd, check=True)

Running: /anaconda/envs/azureml_py310_sdkv2/bin/salmon quant -i /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index -l ISR -1 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_1_trimmed.fastq.gz -2 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/trimmed/SRR13349122_2_trimmed.fastq.gz -p 4 --validateMappings -o /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349122_quant
2026-08-04T15:36:43.657724Z  WARN salmon: --validateMappings has no effect (deprecated in salmon too): selective alignment is the default mapping mode; pass --sketch for pseudoalignment.
2026-08-04T15:36:43.684521Z  INFO piscem_rs::index::reference_index: Loading SSHash dictionary from /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index/index.ssi
2026-08-04T15:36:43.940200Z  INFO piscem_rs::index::reference_index:   k=31, 10472 strings, canonical=true
2026-08-04T15:36:43.940228Z  INFO piscem_rs::index::reference_index: Loading contig table from /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index/index.ctab
2026-08-04T15:36:43.961894Z  INFO piscem_rs::index::reference_index:   10472 contigs, 16626 entries, entry_width=37 bits
2026-08-04T15:36:43.961920Z  INFO piscem_rs::index::reference_index: Loading reference info from /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/reference/transcriptome_index/index.refinfo

### STEP 11: Report the top 10 most highly expressed genes in the samples

Top 10 most highly expressed genes in each wild-type sample.

In [ ]:
!head /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349122_quant/quant.sf -n 1
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349122_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349123_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349124_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349125_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349126_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349127_quant/quant.sf | column -t | sort -k4,4nr 


Top 10 most highly expressed genes in the double lysogen samples.

In [ ]:
! head /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349122_quant/quant.sf -n 1
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349128_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349129_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349130_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349131_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349132_quant/quant.sf | column -t | sort -k4,4nr 
!head -10 /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349133_quant/quant.sf | column -t | sort -k4,4nr 


### STEP 12: Report the expression of a putative acyl-ACP desaturase (BB28_RS16545) that was downregulated in the double lysogen relative to wild-type

A acyl-transferase was reported to be downregulated in the double lysogen as shown in the table of the top 20 upregulated and downregulated genes from the paper describing the study.

Use grep to report the expression in the wild-type sample. The fields in the Salmon quant.sf file are as follows. The level of expression is reported in the Transcripts Per Million (TPM) and number of reads (NumReads) fields:
Name    Length  EffectiveLength TPM     NumReads

In [ ]:
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349122_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349123_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349124_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349125_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349126_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349127_quant/quant.sf

Use grep to report the expression in the double lysogen sample. The fields in the Salmon quant.sf file are as follows. The level of expression is reported in the Transcripts Per Million (TPM) and number of reads (NumReads) fields:
Name    Length  EffectiveLength TPM     NumReads

In [ ]:
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349128_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349129_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349130_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349131_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349132_quant/quant.sf
! grep 'BB28_RS16545' /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/SRR13349133_quant/quant.sf

### STEP 13: Combine Genecounts to a Single Genecount File

Commonly, the readcounts for each sample are combined into a single table, where the rows contain the gene ID, and the columns identify the sample.

In [ ]:
##first merge salmon files by number of reads.
! /anaconda/envs/azureml_py310_sdkv2/bin/salmon quantmerge --column numreads --quants /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/*_quant -o /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt

## we can rename the columns
! sed -i "1s/.*/Name\tSRR13349122\tSRR13349123\tSRR13349124\tSRR13349125\tSRR13349126\tSRR13349127\tSRR13349128\tSRR13349129\tSRR13349130\tSRR13349131\tSRR13349132\tSRR13349133/" /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt

##for further formatting, it may be easier in our r-code to later merge
##if we remove the gene- and rna- prefix
! sed -i "s/gene-//" /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt
! sed -i "s/rna-//" /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt

print("genecount outputfile.")
! head /mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt
   

### STEP 14: Differentially Expressed Genes(DEG) Analysis  
#### STEP 14.1: Install libraries  

In [ ]:
# Install anndata and numeric deps via conda (fast, fewer ABI issues)
%conda install -y anndata>=0.9 numpy pandas scipy statsmodels

In [ ]:
%pip install --upgrade pydeseq2

#### STEP 14.2: Read Data

In [ ]:
import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

#read-in the raw gene count file to a dataframe variable we named 'df'
df = pd.read_csv("/mnt/batch/tasks/shared/LS_root/mounts/clusters/upadhyayk21/code/data/quants/merged_quants.txt", delimiter="\t")

# 1) Replace all zeros with ones
df = df.replace(0, 1)

# 2) Extract columns 2–13 (Python uses 0-based indexing → columns 1:13)
rnaseqMatrix = df.iloc[:, 1:13].round()

# 3) Set rownames using column 1 (ID column)
rnaseqMatrix.index = df.iloc[:, 0]
#rnaseqMatrix.index = rnaseqMatrix.index[1:]
# set first column as index
rnaseqMatrix = df.set_index(df.columns[0])   
rnaseqMatrix = rnaseqMatrix.iloc[:, :].round()
rnaseqMatrix.index.name = None  
rnaseqMatrix_1 = rnaseqMatrix.transpose()
# 4) Preview
print(rnaseqMatrix_1.head())


#### STEP 14.3: Specifying Experimental Design

Next specify the experimental design.

The deseq2 tool will later use this design to group samples together, and output information about the statistical differences in gene expression between these specified groups.

In [ ]:
# define the sample experimental design, in this case 6 wildtype and 6 bacteriophage infected samples
#6 WT + 6 BPs_lysogen
treatment = ["WT"] * 6 + ["BPs_lysogen"] * 6

# build DataFrame
samples = pd.DataFrame({
    "ID": rnaseqMatrix.columns,
    "Treatment": treatment
})

# set rownames equivalent
samples.index = samples["ID"]
samples["Treatment"] = samples["Treatment"].astype("category")
samples


#### STEP 14.4: Creating Deseq2 Object

Now use the treatment design matrix in combination with the readcount matrix to create a deseq2 object.

Once created, this is also a good opportunity to filter out lowly expresseed genes, and to inspect the pre-normalized data using pairwise comparison plots.

Finally, the deseq2 analysis can be run on the deseq2 object.

In [ ]:
# Create DESeq2 dataset object
dds = DeseqDataSet( counts=rnaseqMatrix_1, metadata=samples, design="~Treatment")

# Fit the model
dds.deseq2()

# statistical tests: Wald tests & results for main contrast
# By default, contrast is taken from 'condition' reference vs other levels.
# To set a specific contrast, pass contrast=('condition', 'treated', 'control') etc.

stat_res = DeseqStats(dds, contrast=("Treatment","WT","BPs_lysogen"))
stat_res.summary() 


In [ ]:
head(stat_res.results_df)

#### STEP 14.5: Generating Statistics on data

In [ ]:
# results table: log2FC, p-value, padj, baseMean, etc.
res_df = stat_res.results_df.sort_values("padj", ascending=True)

# 2.6: LFC shrinkage (apeGLM-like)
# This often improves ranking & visualization (similar to DESeq2 recommendation)
stat_res.lfc_shrink(coeff="Treatment[T.WT]")
res_shrunken = stat_res.results_df.sort_values("padj", ascending=True)


In [ ]:
# save DEG results
os.makedirs("results", exist_ok=True)
res_df.to_csv("results/DESeq2_py_results.csv")
res_shrunken.to_csv("results/DESeq2_py_results_shrunken.csv")


In [ ]:
# export top DEGs for downstream enrichment
top_degs = res_df.query("padj < 0.05").sort_values("log2FoldChange", ascending=False)
top_degs.head(50).to_csv("results/top50_upregulated.csv")  # adjust size as needed
top_degs.tail(50).to_csv("results/top50_downregulated.csv")

### Conclusion  

- End to end Bulk RNA-Seq Analysis
  - Fetching SRA files and converting them into FASTQ
  - QC on FastQ
  - Fetching reference genome from NCBI
  - Alignment with Salmon
  - Gene Quantification
  - Differentially expressed genes (DESeq2)

### Cleanup  

Make sure you stop your compute instance and if desired, delete the resource group associated with this tutorial.